# Regression vs Classification

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement simple regression and classification models
- Use metrics (MSE, accuracy) and interpret results

## 🔗 Where this fits

**Builds on:** Unit 2, lesson 06 "Developing Simple Supervised and Unsupervised Learning Models" — the same fit/predict loop, now split by what the target looks like.

**Used later in:** Course 04 (AIAT 114) — Units 1-3, which are this distinction expanded into two full units of algorithms.

---


## 🎯 A $304 million regression error

Zillow, the largest US property website, ran **Zillow Offers**: an algorithmic
home-buying business. A regression model estimated what a house would sell for,
Zillow bought at that price, made light repairs and resold. On **2 November
2021** the board shut it down. In its own filing Zillow recorded an inventory
write-down of about **$304 million** for homes bought in the third quarter at
prices above what it now expected them to fetch, warned of a further $240-265
million in Q4, and announced that roughly **25% of its workforce** would go. The
company's stated reason: "unpredictability in forecasting home prices far
exceed[ed] what we anticipated".

The model was not stupid. It was a house-price regression — the same task, the
same target variable, and very likely a better model than the one you are about
to fit on California census data. What broke was not the fit; it was that a
number produced by fitting the past was used to make an irreversible commitment
about the future, at scale, in a market that had started moving.

### What goes wrong if you pick the wrong side of this split

The two tasks in this notebook are not interchangeable, and neither are their
error measures. Predict a **number** and your error has direction and magnitude:
Zillow's model was wrong by dollars, in one direction, and that is exactly what
cost $304 million. Predict a **category** and error is a count of wrong labels,
and "how wrong" has no meaning — but *which* label you got wrong does, because
missing a cancer and raising a false alarm are not the same event. Choosing the
wrong task type means you spend the whole project optimising a metric that cannot
see the mistake that will actually hurt you.


# Regression vs Classification

**Unit:** Unit 3: AI Concepts, Terminology, and Application Domains Part 2  

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand the difference between regression and classification
- Know when to use each type of problem
- Implement both regression and classification models
- Understand common models and applications

---

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# The two supervised-learning tasks side by side: regression predicts NUMBERS,
# classification predicts CATEGORIES. Which one you have decides the model AND the metric.
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
import matplotlib.pyplot as plt

print("=== Regression vs Classification ===")
print("\nRegression:")
print("  - Predicts continuous values (numbers)")
print("  - Examples: Price, Temperature, Age")
print("  - Output: Real numbers")
print("  - Metrics: MSE, RMSE, MAE, R²")

print("\nClassification:")
print("  - Predicts discrete categories (classes)")
print("  - Examples: Spam/Not Spam, Disease/Healthy, Cat/Dog")
print("  - Output: Class labels")
print("  - Metrics: Accuracy, Precision, Recall, F1-score")

=== Regression vs Classification ===

Regression:
  - Predicts continuous values (numbers)
  - Examples: Price, Temperature, Age
  - Output: Real numbers
  - Metrics: MSE, RMSE, MAE, R²

Classification:
  - Predicts discrete categories (classes)
  - Examples: Spam/Not Spam, Disease/Healthy, Cat/Dog
  - Output: Class labels
  - Metrics: Accuracy, Precision, Recall, F1-score


## Example 1: Regression — Real California House Prices

We use the **California Housing** dataset: 20,640 census block groups from the 1990 US
census, each with a real median house value. Nothing here is invented — these are prices
people actually paid.

In [2]:
# Regression on REAL data: predict the median house value of a California census
# block group from its median income. Target is in units of $100,000.
california = fetch_california_housing()

# Classroom-size the data: a random 2,000-block-group sample keeps the notebook fast.
# random_state=42 makes the sample identical on every machine.
df_house = pd.DataFrame(california.data, columns=california.feature_names)
df_house["MedHouseVal"] = california.target
df_house = df_house.sample(n=2000, random_state=42).reset_index(drop=True)

print(f"Real dataset: California Housing (1990 US census)")
print(f"  Full size: {california.data.shape[0]:,} block groups — we sample 2,000 (random_state=42)")
print(f"  Median income (MedInc) is in tens of thousands of dollars")
print(f"  Target (MedHouseVal) is median house value in hundreds of thousands of dollars")
print(f"\n  Real price range in our sample: "
      f"${df_house['MedHouseVal'].min() * 100_000:,.0f} to ${df_house['MedHouseVal'].max() * 100_000:,.0f}")

# One feature keeps the story simple: does household income predict house value?
X_reg = df_house[["MedInc"]].values
y_reg = df_house["MedHouseVal"].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Train regression model
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

# Predictions
y_pred = reg_model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("\n=== Regression Model Results (real data) ===")
print(f"Model: Linear Regression")
print(f"Mean Squared Error (MSE): {mse:,.4f}   (in units of $100k squared)")
print(f"Root Mean Squared Error (RMSE): {rmse:,.4f}  =  ${rmse * 100_000:,.0f} typical error")
print(f"R² Score: {reg_model.score(X_test, y_test):.4f}")
print(f"\nExample prediction:")
print(f"  A block group with median income $50,000 (MedInc = 5.0)")
print(f"  Predicted median house value: ${reg_model.predict([[5.0]])[0] * 100_000:,.2f}")

print("\n💡 Note the R². Income alone explains only part of real house prices — location,")
print("   house age and rooms matter too. That gap is honest: with invented data you can")
print("   always manufacture an R² near 1.0, and learn nothing from it.")

Real dataset: California Housing (1990 US census)
  Full size: 20,640 block groups — we sample 2,000 (random_state=42)
  Median income (MedInc) is in tens of thousands of dollars
  Target (MedHouseVal) is median house value in hundreds of thousands of dollars

  Real price range in our sample: $30,000 to $500,001

=== Regression Model Results (real data) ===
Model: Linear Regression
Mean Squared Error (MSE): 0.7284   (in units of $100k squared)
Root Mean Squared Error (RMSE): 0.8535  =  $85,346 typical error
R² Score: 0.4734

Example prediction:
  A block group with median income $50,000 (MedInc = 5.0)
  Predicted median house value: $251,561.79

💡 Note the R². Income alone explains only part of real house prices — location,
   house age and rooms matter too. That gap is honest: with invented data you can
   always manufacture an R² near 1.0, and learn nothing from it.


## Example 2: Classification — Real Text from Public Newsgroups

Spam filtering is text classification, so we practise on **real text**: 20 Newsgroups is an
archive of genuine Usenet posts written by real people in the 1990s. We take two topics —
baseball and medicine — and ask a model to tell them apart from the words alone. This is
exactly the machinery a spam filter uses, on messages nobody wrote for a textbook.

In [3]:
# Classification on REAL text: which newsgroup did this post come from?
# We strip headers, quoted replies and signature footers so the model must learn from
# the actual prose rather than from give-away metadata.
categories = ["rec.sport.baseball", "sci.med"]
news_train = fetch_20newsgroups(subset="train", categories=categories,
                                remove=("headers", "footers", "quotes"), random_state=42)
news_test = fetch_20newsgroups(subset="test", categories=categories,
                               remove=("headers", "footers", "quotes"), random_state=42)

print("Real dataset: 20 Newsgroups (genuine Usenet posts)")
print(f"  Training posts: {len(news_train.data)}   Test posts: {len(news_test.data)}")
print(f"  Classes: {news_train.target_names}")
print(f"\n  A real post looks like this (first 300 characters):")
print("  " + repr(news_train.data[0][:300]))

# TF-IDF turns each post into a vector: common words get low weight, distinctive words
# get high weight. min_df=3 ignores words appearing in fewer than 3 posts (real typos).
vectorizer = TfidfVectorizer(stop_words="english", min_df=3, max_features=5000)
X_train_clf = vectorizer.fit_transform(news_train.data)   # learn vocabulary from TRAIN only
X_test_clf = vectorizer.transform(news_test.data)         # reuse that vocabulary on TEST
y_train_clf, y_test_clf = news_train.target, news_test.target

print(f"\n  Vocabulary learned from real posts: {len(vectorizer.vocabulary_):,} words")

# Train classification model
clf_model = LogisticRegression(max_iter=2000)
clf_model.fit(X_train_clf, y_train_clf)

# Predictions
y_pred_clf = clf_model.predict(X_test_clf)

# Evaluate
accuracy = accuracy_score(y_test_clf, y_pred_clf)

print("\n=== Classification Model Results (real text) ===")
print(f"Model: Logistic Regression on TF-IDF features")
print(f"Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_clf, y_pred_clf, target_names=news_train.target_names))

# The words the model leaned on hardest — a sanity check that it learned real topic signal.
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf_model.coef_[0]
print("Most baseball-ish words the model found:", ", ".join(feature_names[np.argsort(coefs)[:8]]))
print("Most medicine-ish words the model found:", ", ".join(feature_names[np.argsort(coefs)[-8:]]))

Real dataset: 20 Newsgroups (genuine Usenet posts)
  Training posts: 1191   Test posts: 793
  Classes: ['rec.sport.baseball', 'sci.med']

  A real post looks like this (first 300 characters):
  'I am looking for a source of American League baseball stats for\nindividual players in the same format as printed in newspapers, ie. I do\nnot want to provide a list of players and get back nice printed reports\nfor $35 a week.\n\nDoes anyone know of such statistics availability and an idea of the\ncost?'

  Vocabulary learned from real posts: 5,000 words

=== Classification Model Results (real text) ===
Model: Logistic Regression on TF-IDF features
Accuracy: 0.9357

Classification Report:
                    precision    recall  f1-score   support

rec.sport.baseball       0.92      0.95      0.94       397
           sci.med       0.95      0.92      0.93       396

          accuracy                           0.94       793
         macro avg       0.94      0.94      0.94       793
      we

## 💬 Discuss

The two printed results: an RMSE of **0.8535** (about **$85,346** typical error)
with **R² = 0.4734** on California house values; **93.57%** accuracy on
newsgroup posts, with baseball recall 0.95 and sci.med recall 0.92.

1. An $85,000 typical error on homes in a $30,000–$500,001 range. **For which
   business is that model useful and for which is it useless?** Compare a bank
   setting a valuation band for a mortgage against Zillow committing to buy the
   house. Same model, same error — say precisely what makes it acceptable in one
   case and catastrophic in the other.
2. The text classifier is a little better at spotting baseball (recall 0.95) than
   medicine (recall 0.92). If this were a real spam filter, the two errors are
   "spam reaches the inbox" and "a real message is deleted unseen". **Which would
   you rather have, and by how much?** Now argue the opposite side as the person
   whose invoice was deleted.
3. Some tasks can be framed either way: "how many days until this machine fails"
   is a regression; "will it fail within 30 days" is a classification of the same
   underlying reality. Pick a problem from your own field, frame it both ways, and
   say which framing you would ship — and what the other framing would have told
   the customer that yours will not.


## Summary: the same split, two different worlds

- **Regression** predicted a **number** (a real California house value) and was scored with
  MSE / RMSE / R² — errors measured in dollars.
- **Classification** predicted a **category** (which newsgroup a real post came from) and was
  scored with accuracy, precision, recall and F1 — errors counted as wrong labels.

Both used real, published data, so both scores are honest: neither task reaches a perfect
score, because real house prices depend on more than income and real people write posts
that stray off topic.

## ⚠️ Where this breaks

**Look at the printed price range: $30,000 to $500,001.** That `$500,001` is not
a coincidence and not the most expensive house in California. The 1990 census
**top-coded** median house value: every block group worth more than half a million
dollars was recorded as exactly $500,001. So the target variable is **censored at
the top**, and no model trained on it can ever predict above that cap, no matter
how expensive the neighbourhood. It will systematically under-predict the priciest
areas — and the R² will not tell you, because the truth it is scored against has
the same ceiling. Finding a defect like this belongs *before* modelling, and it is
exactly the profiling habit from Unit 2, notebook 08.

**The rest of the honest limits:**

- **R² = 0.4734 from one feature is a floor, not a verdict.** Income alone explains
  under half the variation. That is not "linear regression is bad"; it is "we gave
  it one column". It is also not an invitation to keep adding columns until R²
  looks good — R² never decreases when you add features, even useless ones, which
  is why Course 04 teaches adjusted R² and cross-validation.
- **Regression extrapolates confidently and wrongly.** Ask this model about a
  block group with median income far outside the training range and it will return
  a number with no warning attached. Zillow's failure lives in this sentence.
- **This data is from 1990.** Any conclusion about house prices is a conclusion
  about California thirty-five years ago. The pipeline transfers; the coefficients do not.
- **93.57% accuracy is on a nearly balanced two-class problem** (397 vs 396 test
  posts) where the topics barely overlap. Spam filtering is neither balanced nor
  static: spammers change their wording specifically to defeat the model, which is
  a failure mode no held-out test set can measure.
- **TF-IDF sees words, not meaning.** The model learned that certain tokens
  predict certain groups. Rename a baseball team and it is lost; the newsgroup
  metadata had to be stripped precisely because the model would otherwise have
  keyed on the header instead of the prose — a shortcut, not an understanding.
- **The assumption underneath both tasks:** future data comes from the same
  distribution as the training data. Both real cases in this notebook — Zillow
  and spam — are cases where that assumption failed, and it failed for the same
  reason: **the world reacted to the model**.


## 📚 References

1. Cox, D. R. (1958). *The Regression Analysis of Binary Sequences*. Journal of the Royal Statistical Society, Series B, 20(2), 215–242.
2. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.), Chs. 3–4: Linear Regression and Classification. Springer.
3. Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
4. Grinsztajn, L., Flöge, K., Key, O., et al. (2025). *TabPFN-2.5: Advancing the State of the Art in Tabular Foundation Models*. arXiv. <https://arxiv.org/abs/2511.08667>
5. Pfefferle, A., Hog, J., Purucker, L., et al. (2025). *nanoTabPFN: A Lightweight and Educational Reimplementation of TabPFN*. arXiv. <https://arxiv.org/abs/2511.03634>